1.1 Simplification FALC avec LLM (Claude)

In [ ]:
import anthropic

def simplifier_falc(texte_original):
    """
    Simplifie un texte médical selon les normes FALC en utilisant un LLM (Claude)
    
    Args:
        texte_original: Texte médical à simplifier
    
    Returns:
        Texte simplifié en FALC
    """
    
    prompt = f"""
    Tu es expert en simplification de texte FALC (Facile à Lire et à Comprendre).
    
    Tu respecte les règles FALC suivants :
    - Phrases courtes. Chaque phrase nouvelle commence sur une nouvelle ligne
    - Un mot n'est jamais coupé en fin de ligne [⇒ pas de tiret (-) en fin de ligne]
    - Des points (puces ou numéros) sont utilisés pour lister des thèmes ou idées dans une même phrase
    - 1 idée par phrase
    - Vocabulaire simple et courant
    - Vocabulaire constant : utiliser le même mot pour parler de la même chose, tout au long du document
    - Éviter métaphores, abréviations, initiales, acronymes, les expliquer si utilisées dans le texte, 
    pas en note de bas de page
    - Expliquer termes médicaux et concepts difficiles, les expliquer si utilisées dans le texte, 
    pas en note de bas de page
    - Les titres sont courts et annoncent clairement ce qui va suivre
    - S'adresser directement aux personnes en utilisant des mots comme « vous »
    - Utiliser des phrases positives plutôt que négatives (ex. : préférer « Vous devriez rester jusqu’à 
    la fin de la réunion » plutôt que « Vous ne devriez pas partir avant la fin de la réunion »)
    - Utiliser des phrases actives plutôt que des phrases passives (ex. : « Le médecin vous enverra une 
    lettre » plutôt que « Vous recevrez une lettre envoyée par le médecin »)
    - Le texte est toujours aligné à gauche ; il n'est jamais justifié
    - Le texte est aéré (à larges interlignes et à espacement suffisant entre les caractères), 
    avec de larges marges (le texte ne doit pas avoir l'air à l'étroit dans la page)
    - Le texte est sans italique, sans lettrines, sans police à caractères à empattement, 
    à contour ou à ombre portée, et, si possible, sans caractères spéciaux tels que \, &, <, § ou #
    - La ponctuation est simple. Les caractères doivent au moins avoir la taille 14 de la police Arial 
    et bien se détacher sur le fond et ne pas être soulignés
    - Les mots entièrement en majuscules sont à éviter
    - Le style, la police, la mise en forme et le type d'écriture sont identiques tout au long du texte, 
    qui ne doit pas être trop long
    - Les pages sont numérotées (de la manière suivante : « page 2 sur 4 »)

    
    Le texte à simplifier :
    {texte_original}
    
    Simplifie ce texte et :
    - Garde TOUTES les informations médicales importantes
    - Ne change pas le sens médical
    - Reste précis sur les consignes de santé
    - Ne jamais modifier les valeurs numériques, unités, posologies
    """
    
    try:
        # Appel API
        client = anthropic.Anthropic()
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=2000,
            messages=[{"role": "user", "content": prompt}]
        )
            
        texte_simplifie = response.content[0].text


        with open(texte_original+"_falc", 'w', encoding='utf-8') as f:
                f.write(texte_simplifie)

        return texte_simplifie

    except Exception as e:
        print(f"Erreur simplification FALC: {e}")
        return None

1.2 Traduction des fichiers FALC en anglais, russe, arabe, turc avec DeepL et Google Translate
Permet de comparer la qualité des deux services

In [ ]:
pip install googletrans
from googletrans import Translator
import deepl
import time
from typing import Dict, List

# Configuration
from config import DEEPL_API_KEY
def traduire_deepl(texte: str, langue_cible: str) -> str:
    """
    Traduit un texte avec DeepL

    Args:
        texte: Texte à traduire
        langue_cible: Code langue ('EN', 'RU', 'AR', 'TR')

    Returns:
        Texte traduit
    """
    try:
        translator = deepl.Translator(DEEPL_API_KEY)

        # Mapping des codes langues
        langue_map = {
            "en": "EN-GB",  # Anglais britannique
            "ru": "RU",  # Russe
            "ar": "AR",  # Arabe
            "tr": "TR",  # Turc
        }

        langue = langue_map.get(langue_cible.lower())
        if not langue:
            raise ValueError(f"Langue non supportée: {langue_cible}")

        result = translator.translate_text(
            texte,
            target_lang=langue,
            formality="default",
            preserve_formatting=True,
        )

        return result.text

    except Exception as e:
        print(f"Erreur DeepL: {e}")
        return None


def traduire_google(texte: str, langue_cible: str) -> str:
    """
    Traduit un texte avec Google Translate

    Args:
        texte: Texte à traduire
        langue_cible: Code langue ('en', 'ru', 'ar', 'tr')

    Returns:
        Texte traduit
    """
    try:
        translator = Translator()

        # Google Translate utilise des codes langues standard
        result = translator.translate(
            texte, src="fr", dest=langue_cible.lower()  # source français
        )

        return result.text

    except Exception as e:
        print(f"Erreur Google Translate: {e}")
        return None


def traduire_texte_complet(
    texte: str, langues: List[str] = ["en", "ru", "ar", "tr"]
) -> Dict:
    """
    Traduit un texte avec les 2 services pour comparaison

    Args:
        texte: Texte français à traduire
        langues: Liste des langues cibles

    Returns:
        Dictionnaire avec toutes les traductions
        {
            'en': {'deepl': '...', 'google': '...'},
            'ru': {'deepl': '...', 'google': '...'},
            'ar': {'deepl': '...', 'google': '...'},
            'tr': {'deepl': '...', 'google': '...'}
        }
    """
    traductions = {}

    for langue in langues:
        print(f"\nTraduction en {langue.upper()}...")

        traductions[langue] = {}

        # DeepL
        print(f"  → DeepL...")
        trad_deepl = traduire_deepl(texte, langue)
        if trad_deepl:
            traductions[langue]["deepl"] = trad_deepl
            print(f"DeepL: {len(trad_deepl)} caractères")
        else:
            traductions[langue]["deepl"] = None
            print(f"DeepL: échec")

        # Pause pour éviter rate limiting
        time.sleep(1)

        # Google Translate
        print(f"  → Google Translate...")
        trad_google = traduire_google(texte, langue)
        if trad_google:
            traductions[langue]["google"] = trad_google
            print(f"Google: {len(trad_google)} caractères")
        else:
            traductions[langue]["google"] = None
            print(f"Google: échec")

        # Pause entre langues
        time.sleep(1)

    return traductions


def sauvegarder_traductions(
    traductions: Dict, nom_fiche: str, dossier_output: str = "traductions"
):
    """
    Sauvegarde toutes les traductions dans des fichiers séparés

    Args:
        traductions: Dict avec toutes les traductions
        nom_fiche: Nom de la fiche (ex: 'diabete gestationnel')
        dossier_output: Dossier de sortie
    """
    import os

    os.makedirs(dossier_output, exist_ok=True)

    for langue, versions in traductions.items():
        for service, texte in versions.items():
            if texte:
                nom_fichier = f"{nom_fiche}_{langue}_{service}.txt"
                chemin = os.path.join(dossier_output, nom_fichier)

                with open(chemin, "w", encoding="utf-8") as f:
                    f.write(texte)

                print(f"Sauvegardé: {nom_fichier}")


# Exemple d'utilisation
if __name__ == "__main__":
    # Texte exemple en FALC
    texte_falc = """
    Comment équilibrer vos repas avec le diabète
    
    Vous avez du diabète.
    Vous devez faire attention à ce que vous mangez.
    Voici des conseils simples.
    
    1. Mangez 3 repas par jour.
    Ne sautez pas de repas.
    
    2. Mangez des légumes à chaque repas.
    Les légumes sont bons pour vous.
    
    3. Mangez moins de sucre.
    Le sucre fait monter votre glycémie.
    
    4. Buvez de l'eau.
    Évitez les sodas sucrés.
    """

    print("=" * 60)
    print("TRADUCTION MULTILINGUE - Comparaison DeepL vs Google")
    print("=" * 60)

    # Traduire
    traductions = traduire_texte_complet(texte_falc)

    # Sauvegarder
    print("\n" + "=" * 60)
    print("SAUVEGARDE DES TRADUCTIONS")
    print("=" * 60)
    sauvegarder_traductions(traductions, "equilibrer_repas")

    print("\nTerminé !")
    print(
        f"{len(traductions)} langues × 2 services = {len(traductions) * 2} traductions générées"
    )


1.3 Évaluer la qualité des traductions 

In [ ]:
pip install sacrebleu
from sacrebleu import corpus_bleu
from bert_score import score

def evaluer_traduction(reference, hypothese):
    # BLEU score
    bleu = corpus_bleu([hypothese], [[reference]])
    
    # BERTScore (similarité sémantique)
    P, R, F1 = score([hypothese], [reference], lang='fr')
    
    return {'bleu': bleu.score, 'bertscore': F1.mean().item()}

1.4 Générer les fichiers audio avec text-to-speech

In [ ]:
import os
from pathlib import Path
from typing import Dict, Optional


def generer_audio(
    texte: str,
    langue: str,
    nom_fichier: str,
    dossier_output: str = "audio"
) -> Optional[str]:
    """
    gTTS (gratuit, sans API key, mais qualité moindre)
    
    Args:
        texte: Texte à synthétiser
        langue: Code langue ('fr', 'en', 'ru', 'ar', 'tr')
        nom_fichier: Nom du fichier (sans extension)
        dossier_output: Dossier de sortie
    
    Returns:
        Chemin du fichier audio ou None
    """
    try:
        from gtts import gTTS
        
        os.makedirs(dossier_output, exist_ok=True)
        
        # Mapping des codes langues pour gTTS
        langue_map = {
            'fr': 'fr',
            'en': 'en',
            'ru': 'ru',
            'ar': 'ar',
            'tr': 'tr'
        }
        
        lang_code = langue_map.get(langue, 'fr')
        
        # Générer audio
        print(f"Génération audio {langue.upper()} (gTTS)...")
        tts = gTTS(text=texte, lang=lang_code, slow=True)  # slow=True pour FALC
        
        chemin_fichier = os.path.join(dossier_output, f"{nom_fichier}.mp3")
        tts.save(chemin_fichier)
        
        taille = os.path.getsize(chemin_fichier) / 1024
        print(f"Audio créé: {chemin_fichier} ({taille:.1f} KB)")
        
        return chemin_fichier
    
    except Exception as e:
        print(f"Erreur gTTS: {e}")
        return None


def generer_tous_audios(
    traductions: Dict,
    nom_fiche: str,
    service_traduction: str = 'deepl',
    utiliser_gtts: bool = False
) -> Dict[str, str]:
    """
    Génère tous les fichiers audio pour toutes les langues
    
    Args:
        traductions: Dict des traductions {langue: {service: texte}}
        nom_fiche: Nom de la fiche
        service_traduction: 'deepl' ou 'google' (choix de la traduction à utiliser)
        utiliser_gtts: Si True, utilise gTTS au lieu de Google Cloud TTS
    
    Returns:
        Dict {langue: chemin_audio}
    """
    audios = {}
    
    for langue, versions in traductions.items():
        texte = versions.get(service_traduction)
        
        if not texte:
            print(f"Pas de traduction {service_traduction} pour {langue}")
            continue
        
        nom_fichier = f"{nom_fiche}_{langue}_{service_traduction}"
        
        if utiliser_gtts:
            chemin = generer_audio(texte, langue, nom_fichier)
        
        if chemin:
            audios[langue] = chemin
    
    return audios


# Exemple d'utilisation
if __name__ == "__main__":
    # Exemple de traductions (format de sortie de traduction.py)
    traductions_exemple = {
        'en': {
            'deepl': 'How to balance your meals with diabetes\n\nYou have diabetes...',
            'google': 'How to balance your meals with diabetes\n\nYou have diabetes...'
        },
        'ar': {
            'deepl': 'كيفية موازنة وجباتك مع مرض السكري...',
            'google': 'كيفية موازنة وجباتك مع مرض السكري...'
        }
    }
    
    print("=" * 60)
    print("GÉNÉRATION AUDIO TTS")
    print("=" * 60)
    
    try:
        audios = generer_tous_audios(
            traductions_exemple,
            "diabete gestationnel",
            service_traduction='deepl',
            utiliser_gtts=True  # gTTS
        )
        print(f"\n{len(audios)} fichiers audio générés (gTTS)")

1.5 Uploader les fichiers audio sur Dropbox

In [ ]:
pip install dropbox
import os
from typing import Optional, Dict
from pathlib import Path
import dropbox

# Token d'accès
from config import DROPBOX_TOKEN
def upload_dropbox(chemin_fichier: str) -> Optional[str]:
    """
    Upload sur Dropbox et retourne le lien public
    
    Args:
        chemin_fichier: Chemin du fichier à uploader
    
    Returns:
        URL publique du fichier
    """
    try:
        dbx = dropbox.Dropbox(DROPBOX_TOKEN)
        
        nom_fichier = os.path.basename(chemin_fichier)
        chemin_dropbox = f"/audio/{nom_fichier}"
        
        # Upload
        with open(chemin_fichier, 'rb') as f:
            dbx.files_upload(f.read(), chemin_dropbox, mode=dropbox.files.WriteMode.overwrite)
        
        # Créer lien partagé
        shared_link = dbx.sharing_create_shared_link_with_settings(chemin_dropbox)
        
        # Convertir en lien direct (dl=1 au lieu de dl=0)
        url = shared_link.url.replace('dl=0', 'dl=1')
        
        print(f"Uploadé sur Dropbox: {nom_fichier}")
        print(f"URL: {url}")
        
        return url
    
    except Exception as e:
        print(f"Erreur upload Dropbox: {e}")
        return None

1.6 Générer les QR codes

In [ ]:
import os
import qrcode
from typing import Optional, Dict
from pathlib import Path
def creer_qr(
    url: str,
    nom_fichier: str,
    dossier_output: str = "qrcodes",
    taille_box: int = 10,
    bordure: int = 2
) -> Optional[str]:
    """
    Crée un QR code pointant vers l'URL audio
    
    Args:
        url: URL du fichier audio
        nom_fichier: Nom du fichier QR code (sans extension)
        dossier_output: Dossier de sortie
        taille_box: Taille des carrés du QR code (pixels)
        bordure: Épaisseur de la bordure (en carrés)
    
    Returns:
        Chemin du fichier QR code créé
    """
    try:
        os.makedirs(dossier_output, exist_ok=True)
        
        # Créer le QR code
        qr = qrcode.QRCode(
            version=1,  # Taille (1 = plus petit)
            error_correction=qrcode.constants.ERROR_CORRECT_H,  # Haute correction d'erreur
            box_size=taille_box,
            border=bordure,
        )
        
        qr.add_data(url)
        qr.make(fit=True)
        
        # Créer l'image
        img = qr.make_image(fill_color="black", back_color="white")
        
        # Sauvegarder
        chemin_qr = os.path.join(dossier_output, f"{nom_fichier}.png")
        img.save(chemin_qr)
        
        print(f"QR code créé: {chemin_qr}")
        
        return chemin_qr
    
    except Exception as e:
        print(f"Erreur création QR code: {e}")
        return None


def creer_tous_qrcodes(
    audios_urls: Dict[str, str],
    nom_fiche: str,
    service_traduction: str = 'deepl'
) -> Dict[str, str]:
    """
    Crée tous les QR codes pour tous les audios
    
    Args:
        audios_urls: Dict {langue: url_audio}
        nom_fiche: Nom de la fiche
        service_traduction: Service de traduction utilisé
    
    Returns:
        Dict {langue: chemin_qr}
    """
    qrcodes = {}
    
    for langue, url in audios_urls.items():
        nom_fichier = f"qr_{nom_fiche}_{langue}_{service_traduction}"
        
        chemin_qr = creer_qr(url, nom_fichier)
        
        if chemin_qr:
            qrcodes[langue] = chemin_qr
    
    return qrcodes


# Exemple d'utilisation

if __name__ == "__main__":
    print("=" * 60)
    print("UPLOAD AUDIO & GÉNÉRATION QR CODES")
    print("=" * 60)
    
    # Simuler des fichiers audio locaux
    audios_locaux = {
        'en': 'audio/equilibrer_repas_en_deepl.mp3',
        'ru': 'audio/equilibrer_repas_ru_deepl.mp3',
        'ar': 'audio/equilibrer_repas_ar_deepl.mp3',
        'tr': 'audio/equilibrer_repas_tr_deepl.mp3'
    }
    
    # Upload et génération QR codes
    audios_urls = {}
    qrcodes = {}
    
    for langue, chemin_audio in audios_locaux.items():
        # Créer un fichier fictif pour test
        os.makedirs('audio', exist_ok=True)
        Path(chemin_audio).touch()
        
        print(f"\nUpload {langue.upper()}...")
        
        # Upload (méthode locale pour test)
        url = upload_audio(chemin_audio, methode='local', port=8000)
        
        if url:
            audios_urls[langue] = url
            
            # Générer QR code
            nom_qr = f"qr_equilibrer_repas_{langue}_deepl"
            chemin_qr = creer_qr(url, nom_qr)
            
            if chemin_qr:
                qrcodes[langue] = chemin_qr
    
    print("\n" + "=" * 60)
    print("RÉSUMÉ")
    print("=" * 60)
    print(f"{len(audios_urls)} fichiers uploadés")
    print(f"{len(qrcodes)} QR codes générés")
    
    print("\nURLs générées:")
    for langue, url in audios_urls.items():
        print(f"   {langue.upper()}: {url}")


Chaîne de traitement entièrement automatisée

In [ ]:
def traiter_fiche(fiche_originale_path):

    # 1. Lire fiche originale
    with open(fiche_originale_path, 'r') as f:
        texte_original = f.read()
        
    # 2. Simplification FALC
    texte_falc = simplifier_falc(texte_original)

    # 3. Traductions
    langues = ['en', 'ar', 'ru', 'tr']
    traductions = {'fr': texte_falc}
        
    for lang in langues:
        traductions[lang] = traduire_deepl(texte_falc, lang)

    # 4. Génération audio + QR codes
    for lang, texte in traductions.items():
        # Audio
        audio_path = generer_audio(texte, lang)
            
        # Upload audio (Dropbox)
        audio_url = upload_dropbox(audio_path)
            
        # QR code
        qr_path = creer_qr(audio_url, f"qr_{lang}.png")
        
        print(f"QR crée: {qr_path}")
        print(f"Audio crée: {audio_url}")

# Traiter toutes les fiches
for fiche in ['equilibrer_repas.txt', 'injection_insuline.txt', ...]:
    traiter_fiche(fiche)